# 22 Audio Heads Trainer Check

**Purpose:** Check the audio-head trainer step by step (C search, temperature, nudge) before notebook 23 trains the heads.

**Inputs:** Cached BEATs embeddings for `audio_emotion` and `audio_context` (notebook 20).

**Outputs:** Tables only; nothing is saved. The C, temperature and nudge choices are recorded below.

**Splits used:** train only (out-of-fold); val is only counted; test: not used.

In [1]:
import time

import pandas as pd

from woofdata.tasks import task_rows
from woofdecoder.audio import select_layer, softmax
from woofdecoder.config import load_config
from wooftrainer.audio_trainer.audio_heads import (
    balanced_weights,
    choose_C,
    clip_weights,
    fit_temperature,
    make_folds,
    out_of_fold_logits,
    score_probs,
    train_audio_heads,
)
from wooftrainer.audio_trainer.embeddings import attach_labels, load_embeddings
from wooftrainer.settings import TRAINVAL_SPLITS

pd.set_option("display.width", 200)

In [2]:
AUDIO_CONFIG = load_config()["branches"]["audio"]  # configs/pipeline.yaml
ENCODER = AUDIO_CONFIG["encoder"]  # notebook 20
LAYER = AUDIO_CONFIG["layer"]  # notebook 20

## Split Check

Count self-labeled windows and dogs per category in train and val.

In [3]:
for split in TRAINVAL_SPLITS:
    rows = task_rows("audio_context", split)
    mine = rows[rows["source"] == "self_labeled"]
    print(f"\n{split}")
    print(
        mine.groupby("mapped_category")
        .agg(windows=("clip_id", "size"), dogs=("dog_id", "nunique"))
        .to_string()
    )


train
                   windows  dogs
mapped_category                 
alert_aggressive         9     7
attention_seeking       11     9
distressed_lonely       46    25
other                   10     9
playful_excited          9     6

val
                   windows  dogs
mapped_category                 
alert_aggressive         2     2
attention_seeking        4     1
distressed_lonely       29     8
other                    2     1
playful_excited          1     1


## Data and Model Preparation

Load the arousal labels and make the weights and folds.

In [4]:
x, table = load_embeddings("audio_emotion", "train", ENCODER)
table = attach_labels(table, "audio_emotion", "train")
features = select_layer(x, LAYER)

labels = table["arousal"].to_numpy()
weights = balanced_weights(labels, clip_weights(table))
folds = make_folds(labels)

## Optimizing C_arousal Value (v2: self-labeled audio collection on all 5 categories)

Search the regularisation strength C for the arousal head.

In [5]:
best_C, c_table = choose_C(features, labels, weights, folds)
print(c_table.round(3).to_string(index=False), "\nbest C:", best_C)

 log_loss  macro_f1     C
    1.038     0.543 0.000
    0.976     0.562 0.000
    0.922     0.564 0.000
    0.890     0.572 0.001
    0.899     0.575 0.003
    0.968     0.549 0.010
    1.438     0.521 0.100
    2.414     0.515 1.000 
best C: 0.001


**C_arousal value (only with distressed_lonely data self_labeled)**

 log_loss  macro_f1     C  
    1.038     0.543 0.000  
    0.976     0.562 0.000  
    0.922     0.564 0.000  
    0.890     0.572 0.001  
    0.899     0.575 0.003  
    0.968     0.549 0.010  
    1.438     0.521 0.100  
    2.415     0.515 1.000   
best C: 0.001  

- at C = 1.0, the probe default settings show that the prediction is badly over fitting because log loss is super high (2.415)
- at C = 0.001, the scores show the best of both f1 (0.515 --> 0.572) and log_loss (2.415 --> 0.890).

Thus, we can trust the aurosal probabilities and the strong regularisation stopped the memorizing.

## Optimizing T_arousal Value (v2: self-labeled audio collection on all 5 categories)

Fit the arousal temperature on out-of-fold logits.

In [6]:
logits, classes = out_of_fold_logits(features, labels, weights, folds, best_C)
temperature = fit_temperature(logits, labels, classes, weights)
print("temperature:", round(temperature, 2))
print("before:", score_probs(labels, softmax(logits), classes, weights))
print("after:", score_probs(labels, softmax(logits / temperature), classes, weights))

temperature: 0.86
before: {'log_loss': 0.890245704842196, 'macro_f1': 0.5715719412551769}
after: {'log_loss': 0.8871402924443416, 'macro_f1': 0.5715719412551769}


**T_arousal value (only with distressed_lonely data self_labeled)**

temperature: 0.86  
before: {'log_loss': 0.8902433376411953, 'macro_f1': 0.5715719412551769}  
after: {'log_loss': 0.8871376437531784, 'macro_f1': 0.5715719412551769}  

## Full Audio Train Check (v2: self-labeled audio collection on all 5 categories)

Train all heads with `train_audio_heads` and print every table.

In [7]:
start = time.perf_counter()
heads = train_audio_heads(ENCODER, LAYER)
print(f"took {time.perf_counter() - start:.0f}s\n")
print(heads.summary.round(5).to_string(index=False), "\n")
print(heads.tables["C valence"].round(5).to_string(index=False), "\n")
print(heads.tables["C situation"].round(5).to_string(index=False), "\n")
print(heads.tables["nudge"].round(5).to_string(index=False), "\n")
print(
    pd.DataFrame(heads.nudge.weights, index=heads.nudge.classes, columns=heads.nudge.inputs)
    .round(2)
    .to_string()
)

took 125s

     head      C   l2  temperature  before_log_loss  before_macro_f1  after_log_loss  after_macro_f1 balance_sources
  arousal 0.0010           0.85746          0.89025          0.57157         0.88714         0.57157             NaN
  valence 0.0003           0.83085          1.03771          0.46126         1.03606         0.46126             NaN
situation 0.0001 none      1.22660          1.40195          0.39935         1.39395         0.39935            True 

 log_loss  macro_f1       C
  1.07041   0.44608 0.00003
  1.05125   0.45335 0.00010
  1.03771   0.46126 0.00030
  1.03938   0.46833 0.00100
  1.07226   0.46121 0.00300
  1.17957   0.44498 0.01000
  1.85631   0.43337 0.10000
  3.32996   0.42437 1.00000 

 log_loss  macro_f1       C
  1.40532   0.37075 0.00003
  1.40195   0.39935 0.00010
  1.42776   0.41855 0.00030
  1.46519   0.42783 0.00100
  1.50081   0.39714 0.00300
  1.55649   0.41418 0.01000
  1.76646   0.39532 0.10000
  1.94869   0.37130 1.00000 

 log_loss  

**Full Audio Train Check (only with distressed_lonely data self_labeled)**

took 122s  

head C l2 temperature before_log_loss before_macro_f1 after_log_loss after_macro_f1  
arousal 0.001              0.857            0.890            0.572           0.887           0.572  
valence 0.000              0.831            1.038            0.461           1.036           0.461  
situation 0.000  0.01        0.842            1.154            0.519           1.140           0.509   

 log_loss  macro_f1       C  
  1.07041   0.44608 0.00003  
  1.05126   0.45335 0.00010  
  1.03770   0.46126 0.00030  
  1.03939   0.46833 0.00100  
  1.07226   0.46022 0.00300  
  1.17954   0.44401 0.01000  
  1.85590   0.43508 0.10000  
  3.32881   0.42581 1.00000   

 log_loss  macro_f1       C  
  1.23382   0.46326 0.00003  
  1.17646   0.49901 0.00010  
  1.15417   0.51874 0.00030  
  1.16020   0.52905 0.00100  
  1.19430   0.53022 0.00300  
  1.26851   0.51393 0.01000  
  1.48326   0.47135 0.10000  
  1.66748   0.41939 1.00000   

 log_loss  macro_f1    l2  
    1.154     0.519  none  
    1.146     0.509  0.01  
    1.150     0.516   0.1  
    1.154     0.518   1.0  
    1.154     0.519  10.0   

arousal:High arousal:Low arousal:Medium valence:Negative valence:Neutral valence:Positive    
alert_aggressive          -0.11        -0.04            0.00             -0.05            -0.04             -0.05  
attention_seeking         -0.07        -0.09            0.01             -0.08            -0.05             -0.02  
distressed_lonely          0.23         0.25            0.12              0.26             0.22              0.13   
other                     -0.05        -0.05           -0.05             -0.06            -0.06             -0.04  
playful_excited            0.00        -0.08           -0.08             -0.07            -0.07             -0.02  

## Results and decisions

Arousal, from optimizing C_arousal value and T_arousal value, we can see that best C is equal to 0.001, T is around 0.86, and after log loss is around 0.89. The table output shows that same values as the best configurations for arousal.

Valence, after using the nudge, it helps the output probability as log loss went from 1.038 --> 1.036, which also outperforms the mean probing of log loss 1.342.

- Both C searches have their best inside the grid (0.0003 for valence and situation, each with higher log-loss on both sides). The summary shows 0.000 only because of rounding to 3 decimals.
- All three temperatures are about 0.85, with no warnings. Strong regularisation made each head slightly under-confident, and T sharpens it a little.
- Valence at 1.036 is only 6% better than a blind guess. Fine to show in extras as a weak hint, and the app should say so.

Nudged “won” by 0.008 log-loss, well within noise for folds grouped by about 13 distressed dogs, and it lowered macro-F1 by 0.010.

It isn’t using emotion at all. Look at the distressed_lonely row of W: every value is positive (0.12–0.26), whatever the emotion. The other rows are all slightly negative. Since arousal’s 3 probabilities always sum to 1, and valence’s too, a row with roughly equal values adds the same amount to every bark.

Conclusion: emotion doesn’t add usable information to the situation head with today’s data. Emotion stays in extras.
- Does not effect the audio branch probability (Nudge is used only if it beats no nudge by at least 0.01 log loss and does not lower macro f1 score)
- If more data changes this later